In [1]:
import pandas as pd
import numpy as np

In [2]:
!pip install openpyxl

In [3]:
data = pd.read_excel("../dataset/tamilnadu_house_price_dataset_1000.xlsx")

In [4]:
# Quick look at the data
data.head()

,Location,Total_sqft,Bedrooms,Bathrooms,Balcony,Area_type,Availability,Price,Society,Age_of_property,Amenities,Distance_to_landmark,Floors
0,Coimbatore,602,3,2,1,Super built-up,Ready to move,3642258,Palm Grove,2,Gym,8.73,10
1,Salem,2569,5,1,1,Plot area,Under construction,11021852,Sunshine Residency,14,Pool,5.92,10
2,Erode,1893,3,2,1,Built-up,Ready to move,8405079,Green Valley,12,"Security, Lift, Gym",7.50,2
3,Vellore,2696,1,4,0,Plot area,Under construction,12594984,Palm Grove,11,Gym,4.25,10
4,Salem,1685,1,2,0,Built-up,Under construction,7312911,Skyline Apartments,20,"Lift, Pool, Clubhouse",7.61,14


In [5]:
# Just checking column types and nulls
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Location              1000 non-null   object 
 1   Total_sqft            1000 non-null   int64  
 2   Bedrooms              1000 non-null   int64  
 3   Bathrooms             1000 non-null   int64  
 4   Balcony               1000 non-null   int64  
 5   Area_type             1000 non-null   object 
 6   Availability          1000 non-null   object 
 7   Price                 1000 non-null   int64  
 8   Society               1000 non-null   object 
 9   Age_of_property       1000 non-null   int64  
 10  Amenities             1000 non-null   object 
 11  Distance_to_landmark  1000 non-null   float64
 12  Floors                1000 non-null   int64  
dtypes: float64(1), int64(7), object(5)
memory usage: 101.7+ KB


In [6]:
print("Available columns in dataset:")
for col in data.columns:
    print(f"'{col}'")

Available columns in dataset:
'Location'
'Total_sqft'
'Bedrooms'
'Bathrooms'
'Balcony'
'Area_type'
'Availability'
'Price'
'Society'
'Age_of_property'
'Amenities'
'Distance_to_landmark'
'Floors'


In [7]:
# Drop Unnecessary Columns
# Columns we want to remove (if they exist)
dropped_columns = ["Society", "Amenities"]

# Check which of these columns actually exist
existing_columns = [col for col in dropped_columns if col in data.columns]

# Drop only existing ones
data = data.drop(columns=existing_columns)
print("Columns dropped successfully:")
print(existing_columns)


Columns dropped successfully:
['Society', 'Amenities']


In [8]:
print("Current columns after dropping:")
print(data.columns)

Current columns after dropping:
Index(['Location', 'Total_sqft', 'Bedrooms', 'Bathrooms', 'Balcony',
       'Area_type', 'Availability', 'Price', 'Age_of_property',
       'Distance_to_landmark', 'Floors'],
      dtype='object')


In [9]:
# Checking for missing values before moving on
data.isnull().sum()

Location                0
Total_sqft              0
Bedrooms                0
Bathrooms               0
Balcony                 0
Area_type               0
Availability            0
Price                   0
Age_of_property         0
Distance_to_landmark    0
Floors                  0
dtype: int64

In [10]:
# Converting categorical variables
print("Columns before encoding:")
print(data.columns)
data = pd.get_dummies(data, drop_first=True)
print("\nColumns after encoding:")
print(data.columns)

Columns before encoding:
Index(['Location', 'Total_sqft', 'Bedrooms', 'Bathrooms', 'Balcony',
       'Area_type', 'Availability', 'Price', 'Age_of_property',
       'Distance_to_landmark', 'Floors'],
      dtype='object')

Columns after encoding:
Index(['Total_sqft', 'Bedrooms', 'Bathrooms', 'Balcony', 'Price',
       'Age_of_property', 'Distance_to_landmark', 'Floors',
       'Location_Coimbatore', 'Location_Erode', 'Location_Madurai',
       'Location_Salem', 'Location_Thanjavur', 'Location_Thoothukudi',
       'Location_Tirunelveli', 'Location_Trichy', 'Location_Vellore',
       'Area_type_Plot area', 'Area_type_Super built-up',
       'Availability_Under construction'],
      dtype='object')


In [11]:
# Separating features and target
X = data.drop("Price", axis=1)
y = data["Price"]

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

print("\nTarget sample values:")
y.head()

Features (X) shape: (1000, 19)
Target (y) shape: (1000,)

Target sample values:


0     3642258
1    11021852
2     8405079
3    12594984
4     7312911
Name: Price, dtype: int64

In [12]:
# Train/Test Split
from sklearn.model_selection import train_test_split

# 80-20 split is usually fine for now
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42   # fixed seed so results don’t jump around
)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


Feature scaling completed.


In [14]:
# Train the Model
from sklearn.ensemble import RandomForestRegressor
price_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
price_model.fit(X_train_scaled, y_train)
print("Model training completed.")

Model training completed.


In [15]:
# Making predictions on test data
y_predictions = price_model.predict(X_test_scaled)
print("First 5 predictions:")
print(y_predictions[:5])

First 5 predictions:
[ 6279194.795 13681568.865 14649535.41  10866158.725 15360238.52 ]


In [16]:
# Basic Evaluation
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_predictions)
r2 = r2_score(y_test, y_predictions)
print("Mean Absolute Error (₹):", round(mae, 2))
print("R2 Score:", round(r2, 2))

Mean Absolute Error (₹): 1879692.03
R2 Score: 0.64


In [17]:
# Single Sample Prediction Test
# Take one test sample as DataFrame (keep column names)
sample_row_df = X_test.iloc[[0]]   # double brackets = DataFrame

# Scale using the same scaler
sample_row_scaled = scaler.transform(sample_row_df)

# Predict
predicted_price = price_model.predict(sample_row_scaled)
print("Predicted House Price: ₹", round(predicted_price[0], 2))

Predicted House Price: ₹ 6279194.8


In [18]:
#ACTUAL vs PREDICTED
actual_price = y_test.iloc[0]

print("Actual Price: ₹", actual_price)
print("Predicted Price: ₹", round(predicted_price[0], 2))
print("Difference: ₹", round(abs(actual_price - predicted_price[0]), 2))


Actual Price: ₹ 5065131
Predicted Price: ₹ 6279194.8
Difference: ₹ 1214063.8


In [19]:
#Detailed Metrics
from sklearn.metrics import mean_squared_error
import numpy as np

mse = mean_squared_error(y_test, y_predictions)
rmse = np.sqrt(mse)

print("RMSE: ₹", round(rmse, 2))


RMSE: ₹ 3379444.86


In [20]:
#Save the Model
import pickle
with open("../model/house_price_model.pkl", "wb") as file:
    pickle.dump(price_model, file)
print("Model saved successfully.")

Model saved successfully.


In [21]:
import os
if os.path.exists("../model/house_price_model.pkl"):
    print("Model file exists and is ready for deployment.")
else:
    print("Model file not found.")

Model file exists and is ready for deployment.


In [22]:
import pickle
import os

os.makedirs("../model", exist_ok=True)

# Save trained model (correct name)
with open("../model/house_price_model.pkl", "wb") as f:
    pickle.dump(price_model, f)

# Save scaler
with open("../model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save feature column names
with open("../model/feature_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("All files saved successfully!")


All files saved successfully!


In [23]:
#To check no of records
data.shape[0]

1000